# Agent Trust Model Comparison

Compare trust evidence from:

- AgentCensus
- A2A Registry

## Notebook setup

Imports, AgentCensus and A2A Registry credentials, and the shared natural-language query live here. Credentials are loaded from environment variables or the ignored `EXPERIMENTS/notebooks/credentials.yaml` file and are never printed.

In [1]:
import json
import os
import ssl
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlopen

try:
    import EXPERIMENTS.notebooks.utils as experiment_utils
except ModuleNotFoundError:
    import utils as experiment_utils

a2a_registry_load_api_key = experiment_utils.a2a_registry_load_api_key
a2a_registry_print_table = experiment_utils.a2a_registry_print_table
a2a_registry_request_json = experiment_utils.a2a_registry_request_json
agentcensus_get_json = experiment_utils.agentcensus_get_json
agentcensus_load_api_key = experiment_utils.agentcensus_load_api_key
agentcensus_render_integrity_table = experiment_utils.agentcensus_render_integrity_table

AGENTCENSUS_BEARER_TOKEN = agentcensus_load_api_key(required=True)
A2A_REGISTRY_API_TOKEN = a2a_registry_load_api_key(required=True)

SEARCH_QUERY = (
    "Find an A2A agent that measures AI systems"
)
AGENTCENSUS_SEARCH_LIMIT = 20
ANS_DNS_RESOLVER_URL = "https://dns.google/resolve"
ANS_TLS_CONTEXT = ssl.create_default_context(cafile="/etc/ssl/cert.pem")

NOTEBOOK_DIR = Path(experiment_utils.__file__).resolve().parent
OUTPUT_DIR = NOTEBOOK_DIR.parent / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_json(filename, payload):
    path = OUTPUT_DIR / filename
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )
    display_path = Path(os.path.relpath(path, start=Path.cwd()))
    print(display_path)
    return display_path

## Agent search

Assume the caller knows the task but does not already know an agent name, domain, or identifier. Send the same natural-language request to both discovery APIs and keep their complete responses visible.

### Raw AgentCensus search response

`GET /search` performs AgentCensus hybrid retrieval. The complete response is written to `EXPERIMENTS/output/`; the cell prints only its file location. Search rank and match fields describe retrieval relevance, not trust.

In [2]:
agentcensus_search_response = agentcensus_get_json(
    "search",
    {
        "q": SEARCH_QUERY,
        "mechanism": "a2a",
        "limit": AGENTCENSUS_SEARCH_LIMIT,
    },
    token=AGENTCENSUS_BEARER_TOKEN,
)
save_json(
    "trust_model_comparison_agentcensus_search.json",
    agentcensus_search_response,
)

../output/trust_model_comparison_agentcensus_search.json


PosixPath('../output/trust_model_comparison_agentcensus_search.json')

### Raw A2A Registry search response

`GET /public/agents` receives the same natural-language request. The complete registry response is written to `EXPERIMENTS/output/`; the cell prints only its file location.

In [3]:
a2a_registry_search_response = a2a_registry_request_json(
    "GET",
    "/public/agents",
    params={"q": SEARCH_QUERY, "page": 1},
    authenticated=False,
)
save_json(
    "trust_model_comparison_a2a_registry_search.json",
    a2a_registry_search_response,
)

../output/trust_model_comparison_a2a_registry_search.json


PosixPath('../output/trust_model_comparison_a2a_registry_search.json')

### Three agents retained for the comparison

These are concrete examples from the two search results above:

- **Shared:** Council of AI — Measurement Agent appears in both sources on `councilof.ai`.
- **AgentCensus only:** Selnoviktech Agent appears in the AgentCensus results and was not returned by the A2A Registry search above.
- **A2A Registry only:** A2A402 Agent Marketplace appears in the registry results and was not returned by the AgentCensus search above; the later AgentCensus domain read also returns `404 not_found`.

The labels describe what these API checks returned, not proof that an agent does or does not exist elsewhere.

In [4]:
AGENTS = [
    {
        "case": "shared",
        "output_label": "shared_council_of_ai",
        "name": "Council of AI — Measurement Agent",
        "domain": "councilof.ai",
        "agentcensus_agent_key": "ag_0257be5ab061",
        "a2a_registry_package": "ai.councilof.council_of_ai__measurement_agent",
        "a2a_manifest_url": "https://councilof.ai/.well-known/agent.json",
    },
    {
        "case": "agentcensus_only",
        "output_label": "only_selnoviktech",
        "name": "Selnoviktech Agent",
        "domain": "selnoviktech.com",
        "agentcensus_agent_key": "ag_050fb7a9ceea",
        "a2a_registry_package": None,
        "a2a_manifest_url": None,
    },
    {
        "case": "a2a_registry_only",
        "output_label": "missing_a2a402",
        "name": "A2A402 Agent Marketplace",
        "domain": "a2a402.market",
        "agentcensus_agent_key": None,
        "a2a_registry_package": "market.a2a402.a2a402_agent_origin_market",
        "a2a_manifest_url": "https://a2a402.market/.well-known/agent-card.json",
    },
]

a2a_registry_print_table(
    AGENTS,
    [
        "case",
        "name",
        "domain",
        "agentcensus_agent_key",
        "a2a_registry_package",
        "a2a_manifest_url",
    ],
    limit=len(AGENTS),
)

case              | name                              | domain           | agentcensus_agent_key | a2a_registry_package                          | a2a_manifest_url                                 
------------------+-----------------------------------+------------------+-----------------------+-----------------------------------------------+--------------------------------------------------
shared            | Council of AI — Measurement Agent | councilof.ai     | ag_0257be5ab061       | ai.councilof.council_of_ai__measurement_agent | https://councilof.ai/.well-known/agent.json      
agentcensus_only  | Selnoviktech Agent                | selnoviktech.com | ag_050fb7a9ceea       | None                                          | None                                             
a2a_registry_only | A2A402 Agent Marketplace          | a2a402.market    | None                  | market.a2a402.a2a402_agent_origin_market      | https://a2a402.market/.well-known/agent-card.json


## Identity & Control

Identity and control asks **who operates an agent** and **whether control has been proved**.

* The available AgentCensus calls are public reads. Their agent and domain records provide identity context, but they do not prove that a real-world organization controls the agent or domain.
* Treat operator names, domains, endpoints, and published metadata as claims unless they are supported by independently verifiable control evidence.

### AgentCensus: raw public agent records

`GET /agents/{agentKey}` returns the published identity, discovery provenance, observed status, and related raw fields for one census record. Each complete response is written to `EXPERIMENTS/output/`; the cell prints only the file locations.

In [5]:
agentcensus_agent_identity_responses = {}
for agent in AGENTS:
    agent_key = agent["agentcensus_agent_key"]
    if agent_key is None:
        continue
    agentcensus_agent_identity_responses[agent_key] = agentcensus_get_json(
        f"agents/{agent_key}", token=AGENTCENSUS_BEARER_TOKEN
    )
    save_json(
        f"trust_model_comparison_agentcensus_{agent['output_label']}_agent.json",
        agentcensus_agent_identity_responses[agent_key],
    )

../output/trust_model_comparison_agentcensus_shared_council_of_ai_agent.json
../output/trust_model_comparison_agentcensus_only_selnoviktech_agent.json


### AgentCensus: raw public domain records

`GET /domains/{domain}` returns the domain-level record and observations. Each complete response is written to `EXPERIMENTS/output/`; the cell prints only the file locations. A domain record is not, by itself, proof that the agent is controlled by a named organization.

In [6]:
agentcensus_domain_identity_responses = {}
for agent in AGENTS:
    domain = agent["domain"]
    agentcensus_domain_identity_responses[domain] = agentcensus_get_json(
        f"domains/{domain}", token=AGENTCENSUS_BEARER_TOKEN
    )
    save_json(
        f"trust_model_comparison_agentcensus_{agent['output_label']}_domain.json",
        agentcensus_domain_identity_responses[domain],
    )

../output/trust_model_comparison_agentcensus_shared_council_of_ai_domain.json
../output/trust_model_comparison_agentcensus_only_selnoviktech_domain.json
../output/trust_model_comparison_agentcensus_missing_a2a402_domain.json


### A2A Registry: raw public identity records

The relevant public registry records are already present in the raw `GET /public/agents` search response. This cell preserves each selected A2A Registry record unchanged in its own JSON file and prints only the file locations. Registry verification fields are identity claims from A2A Registry; search relevance and health fields are not identity evidence.

In [7]:
a2a_registry_identity_search_response = json.loads(
    (OUTPUT_DIR / "trust_model_comparison_a2a_registry_search.json").read_text(
        encoding="utf-8"
    )
)
a2a_registry_agents = a2a_registry_identity_search_response["data"]["agents"]
a2a_registry_selected_packages = {
    "ai.councilof.council_of_ai__measurement_agent": "shared_council_of_ai",
    "market.a2a402.a2a402_agent_origin_market": "missing_a2a402",
}
for package_name, output_label in a2a_registry_selected_packages.items():
    a2a_registry_agent = next(
        item for item in a2a_registry_agents
        if item["packageName"] == package_name
    )
    save_json(
        f"trust_model_comparison_a2a_registry_{output_label}_agent.json",
        a2a_registry_agent,
    )

../output/trust_model_comparison_a2a_registry_shared_council_of_ai_agent.json
../output/trust_model_comparison_a2a_registry_missing_a2a402_agent.json


### ANS: public identity discovery

ANS verification starts with the domain's `_ans-badge` TXT record, with `_ra-badge` as the legacy fallback. This cell preserves the complete DNS-over-HTTPS responses for the same three domains in one JSON file and prints only its location. A missing record means no public ANS badge was discovered; it is not a failed ANS verification.

In [8]:
ans_selected_domains = {
    "shared_council_of_ai": "councilof.ai",
    "only_selnoviktech": "selnoviktech.com",
    "missing_a2a402": "a2a402.market",
}
ans_identity_discovery = []
for output_label, domain in ans_selected_domains.items():
    for prefix in ("_ans-badge", "_ra-badge"):
        record_name = f"{prefix}.{domain}"
        request_url = f"{ANS_DNS_RESOLVER_URL}?{urlencode({'name': record_name, 'type': 'TXT'})}"
        with urlopen(request_url, timeout=30, context=ANS_TLS_CONTEXT) as response:
            ans_identity_discovery.append({
                "outputLabel": output_label,
                "domain": domain,
                "recordName": record_name,
                "url": request_url,
                "httpStatus": response.status,
                "data": json.loads(response.read().decode("utf-8")),
            })
save_json(
    "trust_model_comparison_ans_identity_discovery.json",
    ans_identity_discovery,
)

../output/trust_model_comparison_ans_identity_discovery.json


### 🍀 Actual useful Identity & Control evidence

Only fields that help identify the agent, associate it with an operator record, or show whether control was verified are included. A blank cell means the agent was not found in that source. `--` means ANS provided no useful evidence for the selected agent because no public ANS badge was discovered.

| Agent | Source | Actual Useful Evidence | Reasons |
|---|---|---|---|
| Council of AI — Measurement Agent | AgentCensus | • `domain = councilof.ai`<br>• `endpointHost = councilof.ai`<br>• `endpointSameOrigin = false`<br>• `activeVerification = null` | The domain and endpoint host establish a technical identity association. However, the negative same-origin observation and absent active verification provide no proof that an operator controls them. |
| Council of AI — Measurement Agent | A2A Registry | • `orgId = org_system_unclaimed`<br>• `verification_level = unclaimed`<br>• `isVerified = false` | These fields directly show that no organization has claimed this registry record and that the registry has not verified its controller. |
| Council of AI — Measurement Agent | ANS | -- | No public ANS badge was discovered, so ANS supplies no identity or control evidence. |
| Selnoviktech Agent | AgentCensus | • `domain = selnoviktech.com`<br>• `endpointHost = selnoviktech.com`<br>• `endpointSameOrigin = true`<br>• `activeVerification = null` | Same-origin alignment ties the observed endpoint to the claimed domain, which supports technical identity continuity. It does not prove operator control because active verification is absent. |
| Selnoviktech Agent | A2A Registry |  |  |
| Selnoviktech Agent | ANS | -- | No public ANS badge was discovered, so ANS supplies no identity or control evidence. |
| A2A402 Agent Marketplace | AgentCensus |  |  |
| A2A402 Agent Marketplace | A2A Registry | • `orgId = HWEFZBB4UVzDhkGBZIjtGdIlihvEf9JP`<br>• `verification_level = verified`<br>• `isVerified = true` | The record is associated with a specific registry organization and carries a positive registry verification assertion. This is useful control evidence, although the output does not expose the verification method. |
| A2A402 Agent Marketplace | ANS | -- | No public ANS badge was discovered, so ANS supplies no identity or control evidence. |

## Integrity

Integrity asks whether an agent's discovery records, versions, endpoints, and signatures remain consistent. Start with AgentCensus and preserve its observations without creating a derived summary or score.

### AgentCensus: raw discovery-document records

The complete `GET /agents/{agentKey}` and `GET /domains/{domain}` responses saved above already contain AgentCensus history, provenance, TLS, and DNS observations. This cell adds the complete `GET /agents/{agentKey}/documents/{source}` response for every discovery source reported by each selected AgentCensus record. AgentCensus returns a normalized parsed snapshot rather than the original fetched bytes. Each response is written unchanged to `EXPERIMENTS/output/`; the cell prints only its file location.

In [9]:
for agent in AGENTS:
    agent_key = agent["agentcensus_agent_key"]
    if agent_key is None:
        continue
    for source in agentcensus_agent_identity_responses[agent_key]["data"]["mechanisms"]:
        response = agentcensus_get_json(
            f"agents/{agent_key}/documents/{source}",
            token=AGENTCENSUS_BEARER_TOKEN,
        )
        save_json(
            f"trust_model_comparison_agentcensus_{agent['output_label']}_{source}_document.json",
            response,
        )

../output/trust_model_comparison_agentcensus_shared_council_of_ai_a2a_document.json
../output/trust_model_comparison_agentcensus_shared_council_of_ai_a2a_alt_document.json
../output/trust_model_comparison_agentcensus_only_selnoviktech_a2a_alt_document.json
../output/trust_model_comparison_agentcensus_only_selnoviktech_oauth_protected_resource_document.json
../output/trust_model_comparison_agentcensus_only_selnoviktech_a2a_document.json
../output/trust_model_comparison_agentcensus_only_selnoviktech_ard_document.json


In [10]:
from IPython.display import HTML

HTML(
    agentcensus_render_integrity_table(
        AGENTS, agentcensus_agent_identity_responses, OUTPUT_DIR
    )
)